In [1]:
import dill
from qiskit.circuit.library import QuantumVolume
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager, PassManager, StagedPassManager
from qiskit.transpiler.passes.scheduling import ALAPScheduleAnalysis, PadDynamicalDecoupling, ASAPScheduleAnalysis
from qiskit_ibm_runtime.fake_provider import FakeBrisbane 
from qiskit.providers.backend import BackendV2 as Backend
from qiskit_ibm_runtime.sampler import SamplerOptions, SamplerV2 as Sampler
from qiskit_ibm_runtime import Batch, QiskitRuntimeService
from qiskit.visualization import plot_histogram
from qiskit.circuit.library import XGate, YGate
from qiskit.quantum_info import hellinger_fidelity
import numpy as np
import pandas as pd 
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.circuit.equivalence_library import (SessionEquivalenceLibrary as sel,)
from qiskit.transpiler.passes import BasisTranslator
from qiskit_ibm_runtime import IBMBackend
from qiskit.visualization import timeline_drawer
from qiskit.transpiler import Target

In [2]:
## Import Necessary packages here. If you are running the notebook from first cell. Leave it blank
import ast
import pandas as pd
from qiskit.visualization import plot_histogram

qubits = [7, 8, 9, 10, 17, 18, 19, 20, 27, 28, 29, 30]


methods_list    = ["none", "asap", "alap"] 
dd_methods_list = ["asap+dd+xx", "alap+dd+xx", "alap+dd+xpxm", "asap+dd+xpxm"]
extra_dd_methods_list = ["asap+dd+xy", "alap+dd+xy"]

valid_methods = methods_list + dd_methods_list + extra_dd_methods_list


In [3]:
# Read the CSV file
df = pd.read_csv("Fez-GHZ-extensive-benchmarking-sampler.csv")

# List of columns that contain string representations of dictionaries
# Based on the columns shown in your traceback
cols_to_eval = df.columns

def safe_literal_eval(val):
    """
    Safely evaluate a string containing a Python literal.
    Returns the original value if it's not a string or if evaluation fails.
    """
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return val
    return val

# Apply the conversion only to the relevant columns
for col in cols_to_eval:
    if col in df.columns:
        df[col] = df[col].apply(safe_literal_eval)

# Verify the data types
df.head()


,iteration,job_id,Qubit,shots,ideal_counts,simulated_counts,none,asap,alap,asap+dd+xx,alap+dd+xx,alap+dd+xpxm,asap+dd+xpxm,asap+dd+xy,alap+dd+xy,hellinger_fidelity(ideal),hellinger_fidelity(simulated)
0,0,d4slsksfitbs739j43k0,7,4096,"{'1111111': 2048, '0000000': 2048}","{'1111111': 2153, '0000000': 1943}","{'0000000': 1928, '0100000': 15, '1111111': 16...","{'1111111': 1665, '0000000': 1879, '0000010': ...","{'0000000': 1907, '1111111': 1672, '0001111': ...","{'0000000': 1805, '1011000': 1, '1111111': 163...","{'1111111': 1602, '0000000': 1857, '0000001': ...","{'1111111': 1615, '0000000': 1880, '0000011': ...","{'0000000': 1833, '1111111': 1622, '1100000': ...","{'1111111': 1642, '0000000': 1892, '0111111': ...","{'1111111': 1707, '0000000': 1827, '0010111': ...","[0.9993424263882965, 0.8767808045734468, 0.864...","[0.8745778053258384, 0.8625377175627317, 0.870..."
1,0,d4slsls5fjns73d2jje0,8,4096,"{'11111111': 2048, '00000000': 2048}","{'11111111': 2074, '00000000': 2022}","{'11111111': 1573, '00000000': 1931, '11101111...","{'11111111': 1564, '00000000': 1832, '00111111...","{'00000000': 1882, '11111111': 1671, '11011111...","{'00000000': 1823, '01000000': 14, '11111111':...","{'00000000': 1856, '11111111': 1595, '11110111...","{'11011101': 1, '11111111': 1628, '00000000': ...","{'00000000': 1790, '11111111': 1588, '11111011...","{'11100001': 1, '00000000': 1816, '11111111': ...","{'11111111': 1602, '00000000': 1896, '11111100...","[0.9999597056364962, 0.8532304421185956, 0.827...","[0.8526413518986303, 0.8273600489160481, 0.866..."
2,0,d4slsmbher1c73bdfi7g,9,4096,"{'111111111': 2048, '000000000': 2048}","{'111111111': 2070, '000000000': 2026}","{'111111111': 1355, '000000000': 1878, '111111...","{'111111111': 1325, '000000000': 1748, '110000...","{'111011111': 48, '111111111': 1427, '00000000...","{'111111111': 1315, '000000000': 1808, '111111...","{'101111111': 22, '111111111': 1364, '11111111...","{'111111110': 224, '000000000': 1846, '1111111...","{'000000000': 1766, '111111111': 1364, '111111...","{'000000000': 1803, '000000001': 57, '00100000...","{'000000000': 1847, '111111111': 1414, '011111...","[0.9999711505196361, 0.7841085083810232, 0.746...","[0.783400226178636, 0.7460971830501215, 0.7981..."
3,0,d4slsmsfitbs739j43n0,10,4096,"{'1111111111': 2048, '0000000000': 2048}","{'1111111111': 2017, '0000000000': 2079}","{'0000000000': 1874, '1111111111': 1337, '1011...","{'1010000000': 6, '1011011111': 4, '1111111111...","{'1111111111': 1368, '0000000000': 1898, '1011...","{'0000000000': 1707, '1111111111': 1373, '0000...","{'0000100000': 40, '0000000000': 1762, '111111...","{'1111111111': 1354, '0000000000': 1853, '1000...","{'0000111111': 21, '1111111111': 1353, '110000...","{'1111111111': 1365, '0000000000': 1721, '0111...","{'0000000000': 1851, '1011111111': 202, '11111...","[0.9999427166549892, 0.7784153151055028, 0.754...","[0.7793632796113626, 0.7556117094632328, 0.793..."
4,0,d4slsnk5fjns73d2jjgg,17,4096,"{'11111111111111111': 2048, '00000000000000000...","{'00000000000000000': 2081, '11111111111111111...","{'11111101111111111': 152, '11111111000000000'...","{'00000000000000000': 1432, '11111111110111111...","{'00000000000000000': 1759, '11111111111111111...","{'11100000000000000': 21, '00000000000000000':...","{'00000000000000111': 23, '11111111111111111':...","{'11111100111111111': 1, '00000000000000000': ...","{'11111000000011111': 2, '00000000000000000': ...","{'00000000000000000': 1314, '11111110111111111...","{'00000000000000000': 1461, '00000000000000010...","[0.9999350863280547, 0.6503878072638585, 0.544...","[0.651677897221708, 0.5453979926742467, 0.6634..."


In [7]:
df_iter = df[df["iteration"] == 0]

In [ ]:
df_iter[df_iter["Qubit"] == 7]["hellinger_fidelity(ideal)"]

,iteration,job_id,Qubit,shots,ideal_counts,simulated_counts,none,asap,alap,asap+dd+xx,alap+dd+xx,alap+dd+xpxm,asap+dd+xpxm,asap+dd+xy,alap+dd+xy,hellinger_fidelity(ideal),hellinger_fidelity(simulated)
0,0,d4slsksfitbs739j43k0,7,4096,"{'1111111': 2048, '0000000': 2048}","{'1111111': 2153, '0000000': 1943}","{'0000000': 1928, '0100000': 15, '1111111': 16...","{'1111111': 1665, '0000000': 1879, '0000010': ...","{'0000000': 1907, '1111111': 1672, '0001111': ...","{'0000000': 1805, '1011000': 1, '1111111': 163...","{'1111111': 1602, '0000000': 1857, '0000001': ...","{'1111111': 1615, '0000000': 1880, '0000011': ...","{'0000000': 1833, '1111111': 1622, '1100000': ...","{'1111111': 1642, '0000000': 1892, '0111111': ...","{'1111111': 1707, '0000000': 1827, '0010111': ...","[0.9993424263882965, 0.8767808045734468, 0.864...","[0.8745778053258384, 0.8625377175627317, 0.870..."
